# Foods experiment

This notebook uses OpenNutrition Foods dataset.

Credit to https://www.opennutrition.app

Here is the download link: https://www.opennutrition.app/download

Let's import the necessary libraries:

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams, RecommendQuery, RecommendInput, RecommendStrategy

load_dotenv("../../_starter/.env")

True

# Initialize the clients

In [2]:
embedding_client = OpenAI(
    base_url = os.getenv("AZURE_COGNITIVE_ENDPOINT"),
    api_key = os.getenv("AZURE_COGNITIVE_KEY"),
)

qdrant_client = QdrantClient(url="http://localhost:6333")

# Data set

In [11]:
# Load opennutrition_foods.tsv
df = pd.read_csv("opennutrition_foods.tsv", sep="\t")

# Keep only type == everyday
df = df[df["type"] == "everyday"]
df = df[["name", "description"]]
df.reset_index(drop=True, inplace=True)
df.head()

,name,description
0,"Chicken Breast, Boneless Skinless, Cooked","Chicken breast (boneless, skinless) is a lean ..."
1,Large Eggs,Large eggs are a common size designation for c...
2,"Rice, Cooked",Cooked rice is a staple grain that serves as a...
3,Whole Milk,Whole milk is a dairy beverage retaining its n...
4,Enriched White Rice,Enriched white rice is a milled grain that has...


In [10]:
def get_vector(text):
    response = embedding_client.embeddings.create(
        input=text,
        model="text-embedding-3-large", # Make sure to use the same model as the one used to create the vectors in the previous example
    )
    return response.data[0].embedding

# Create and add points to collection

In [ ]:
qdrant_client.create_collection(
    collection_name="OpenNutritionFoods",
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)

In [16]:
for idx, row in df.iterrows():
    point = PointStruct(
        id=idx,
        vector=get_vector(row['description']),
        payload={
                "name": row['name'],
                "description": row['description']
        }
    )

    qdrant_client.upsert(
        collection_name="OpenNutritionFoods",
        points=[point]
    )

# Default behaviour

In [22]:
query_vector = get_vector("apple") 

qdrant_client.search(
    collection_name="OpenNutritionFoods",
    query_vector=query_vector,
    limit=20,
)

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_61910/851806458.py:3: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_client.search(


[ScoredPoint(id=32, version=32, score=0.43281853, payload={'name': 'Apples', 'description': 'Apples are a widely cultivated pomaceous fruit belonging to the rose family, available in thousands of varieties ranging from sweet to tart. Their crisp flesh provides dietary fiber (notably pectin) and vitamin C, with most nutrients concentrated in the skin. This low-calorie fruit is typically consumed raw with the peel intact, though it also serves as a versatile ingredient in both sweet and savory cooked dishes.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3518, version=3518, score=0.41220343, payload={'name': 'Apple Baby Food', 'description': 'Apple baby food is a smooth, pureed preparation of cooked apples specifically formulated for infants beginning solid foods. This single-ingredient puree is typically strained to eliminate any chunks or fibrous material, making it safe for early eaters while retaining natural fruit sugars and nutrients. Commercial varieties are ca

# Vector reversal



In [23]:
query_vector = get_vector("apple") 

query_vector = [-x for x in query_vector]

qdrant_client.search(
    collection_name="OpenNutritionFoods",
    query_vector=query_vector,
    limit=20,
)

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_61910/133699966.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_client.search(


[ScoredPoint(id=2471, version=2471, score=0.022847576, payload={'name': 'Miso Soup with Noodles', 'description': 'Miso soup with noodles is a traditional Japanese dish featuring a savory broth made from fermented soybean paste (miso), combined with cooked noodles such as udon or soba. The soup often includes tofu, seaweed, and scallions, providing a balance of plant-based protein, carbohydrates, and gut-friendly probiotics from the miso’s fermentation process. While naturally rich in umami flavor, its sodium content can vary significantly depending on the broth preparation, making portion awareness important for sodium-sensitive diets.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2712, version=2712, score=0.01207895, payload={'name': 'Miso Broth', 'description': 'Miso broth is a traditional Japanese soup base made by dissolving fermented soybean paste (miso) in dashi stock, creating a savory, umami-rich liquid. This broth contains beneficial probiotics from the mi

In [24]:
what_i_ate_last_week = "rice, paneer, curry, water, juice, tea, pizza, bread, eggs, sweets, cheese, pepper, salt, sugar, biscuits, taco, beans, salad"

query_vector = get_vector(what_i_ate_last_week) 

query_vector = [-x for x in query_vector]

qdrant_client.search(
    collection_name="OpenNutritionFoods",
    query_vector=query_vector,
    limit=20,
)

/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_61910/341062768.py:7: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_client.search(


[ScoredPoint(id=3927, version=3927, score=-0.069110945, payload={'name': 'Beaujolais', 'description': 'Beaujolais is a light-bodied red wine produced in eastern France using Gamay grapes, recognized for its bright red fruit flavors and floral aromas. Its signature freshness comes from carbonic maceration fermentation, a process that preserves fruitiness while yielding moderate alcohol levels (typically 12-13% ABV). Distinct from heavier red wines, it is often served slightly chilled and pairs flexibly with dishes ranging from charcuterie to roasted meats.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4918, version=4918, score=-0.07051282, payload={'name': 'Mourvèdre Wine', 'description': 'Mourvèdre (also known as Monastrell) is a full-bodied red wine characterized by its rich dark fruit flavors and prominent tannin structure. This varietal typically contains 14-15% alcohol by volume and exhibits intense notes of blackberry, black pepper, and earthy undertones. Orig

# Subtract to get diet suplements

In [30]:
what_i_ate_last_week = "raspberries, beef, rice, paneer, curry, water, juice, tea, pizza, bread, eggs, sweets, cheese, pepper, salt, sugar, biscuits, taco, beans, salad"

vector_1 = get_vector("healthy benefitial food")
vector_2 = get_vector(what_i_ate_last_week)

In [31]:
# Subtract the fruit vector from the apple vector
queen_maybe = [x-y for x, y in zip(vector_1, vector_2)]

# Perform a similarity search on the collection
results = qdrant_client.search(
    collection_name="OpenNutritionFoods",
    query_vector=queen_maybe,
    limit=10,
)

print(results[0].score - results[1].score)

results

0.0007931599999999872


/var/folders/dp/lpgxnt354y995wh87y8rjr780000gn/T/ipykernel_61910/1083388644.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


[ScoredPoint(id=1417, version=1417, score=0.1796929, payload={'name': 'Egg Yolk', 'description': "Egg yolk is the nutrient-dense yellow portion of a hen's egg, containing nearly all of the egg's fat-soluble vitamins and minerals. It provides significant cholesterol alongside essential nutrients like choline for brain function and vitamin D for bone health, while its viscous texture acts as a natural emulsifier in sauces and custards. Though historically scrutinized for cholesterol content, current nutritional research recognizes yolks as a source of bioavailable nutrients that can fit into balanced diets when consumed moderately."}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2691, version=2691, score=0.17889974, payload={'name': 'Halibut Fillet with Skin, Raw', 'description': 'Halibut fillet with skin is a lean, cold-water flatfish prized for its firm texture and mild, slightly sweet flavor. The edible skin crisps when pan-seared or grilled, while the flesh provide